In [1]:
import yfinance as yf
import duckdb
import pandas as pd
import time
import os

# 1. Ticker Setup
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "NVDA", "TSLA", "META", "BRK-B", "UNH", "JNJ", 
           "V", "XOM", "JPM", "PG", "MA", "LLY", "CVX", "HD", "ABBV", "PFE"] 

db_name = "stock_data_final.db"
if os.path.exists(db_name): os.remove(db_name)
con = duckdb.connect(db_name)

def get_data_safely(symbol):
    try:
        tk = yf.Ticker(symbol)
        
        # --- TABLE 1: Companies (Metadata) ---
        info = tk.info
        df_comp = pd.DataFrame([{
            "symbol": symbol,
            "name": info.get("longName"),
            "sector": info.get("sector"),
            "summary": info.get("longBusinessSummary", "N/A") # Important for file size
        }])

        # --- TABLE 2: PriceHistory ---
        df_h = tk.history(period="max").reset_index()
        df_h['symbol'] = symbol
        # Force Volume to INT64 to prevent overflow errors
        df_h['Volume'] = df_h['Volume'].astype('int64')

        # --- TABLE 3: Fundamentals (Entity-Attribute-Value / Long Format) ---
        f_list = []
        # Pulling Income Statement, Balance Sheet, and Cash Flow
        stmts = {"Income": tk.quarterly_financials, 
                 "Balance": tk.quarterly_balance_sheet, 
                 "Cashflow": tk.quarterly_cashflow}
        
        for stmt_name, df in stmts.items():
            if df is not None and not df.empty:
                # If yfinance returns a Series, convert to DataFrame
                if isinstance(df, pd.Series):
                    df = df.to_frame()
                
                # Transform: Wide -> Long
                # This creates a row for every single metric for every single quarter
                temp_df = df.stack().reset_index()
                temp_df.columns = ['metric', 'report_date', 'value']
                temp_df['symbol'] = symbol
                temp_df['statement_type'] = stmt_name
                f_list.append(temp_df)
        
        df_fund = pd.concat(f_list) if f_list else pd.DataFrame()

        # --- TABLE 4: TechnicalIndicators ---
        df_ind = df_h[['Date', 'symbol', 'Close']].copy()
        # Adding multiple features increases the data density
        for window in [20, 50, 200]:
            df_ind[f'SMA_{window}'] = df_ind['Close'].rolling(window=window).mean()
        df_ind['Daily_Return'] = df_ind['Close'].pct_change()
        
        return df_comp, df_h, df_fund, df_ind
    except Exception as e:
        print(f"Error on {symbol}: {e}")
        return None, None, None, None

# 2. Ingestion Loop
print("Starting heavy data pull. This creates millions of rows...")
for i, s in enumerate(tickers):
    c, h, f, ind = get_data_safely(s)
    
    if h is not None:
        # Save to DuckDB (using 'append' handles the first-time creation automatically)
        c.to_sql('Companies', con, if_exists='append', index=False)
        h.to_sql('PriceHistory', con, if_exists='append', index=False)
        if not f.empty:
            f.to_sql('Fundamentals', con, if_exists='append', index=False)
        ind.to_sql('TechnicalIndicators', con, if_exists='append', index=False)
        
        if i % 5 == 0:
            size_mb = os.path.getsize(db_name) / 1e6
            print(f"Processed {i+1}/{len(tickers)}. DB Size: {size_mb:.2f} MB")
    
    time.sleep(0.2)

# 3. Export to Parquet
print("Exporting tables to Parquet...")
for t in ["Companies", "PriceHistory", "Fundamentals", "TechnicalIndicators"]:
    con.execute(f"COPY {t} TO '{t.lower()}.parquet' (FORMAT PARQUET)")

print(f"Final Database Size: {os.path.getsize(db_name) / 1e9:.2f} GB")
con.close()

Starting heavy data pull. This creates millions of rows...


/tmp/ipykernel_36961/4152997064.py:76: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  c.to_sql('Companies', con, if_exists='append', index=False)
/tmp/ipykernel_36961/4152997064.py:77: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  h.to_sql('PriceHistory', con, if_exists='append', index=False)


TransactionException: TransactionContext Error: cannot rollback - no transaction is active